In [ ]:
import json
import os
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import dotenv_values

In [ ]:
config = dotenv_values(".env")
if not config:
    config = dict(os.environ)

team_id = int(str(config["TEAM_ID"]))
team_name = str(config["TEAM_NAME"])
auth = (str(config["HOLDSPORT_USERNAME"]), str(config["HOLDSPORT_PASSWORD"]))

date = "2024-01-01"

In [ ]:
url = f"https://api.holdsport.dk/v1/teams/{team_id}/activities?date={date}page=1&per_page=120"
headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, auth=auth, timeout=30)
response_dict = json.loads(response.text)

In [ ]:
def get_attending_player_number(activities_users: list[dict[str, any]]) -> int:  # noqa: D103
    attending_players = 0
    for player in activities_users:
        if player["status"] == "Attending":
            attending_players += 1
    return attending_players

In [ ]:
data = []

for response_entry in response_dict:
    activity_name = "Motion A practice"
    start_time = str(response_entry["starttime"])

    if response_entry["name"] == activity_name:
        entry = {
            "date": datetime.strptime(start_time, "%Y-%m-%dT%H:%M:%S%z").date(),
            "attending": get_attending_player_number(response_entry["activities_users"]),
        }
        data.append(entry)

In [ ]:
df = pd.DataFrame(data)  # noqa: PD901
df.set_index("date", inplace=True)  # noqa: PD002

In [ ]:
df["attending"].plot.bar()
plt.ylabel("Number of registered players")
plt.grid(axis="y")
plt.show()